# TP53 Mutation Status Baseline

This notebook recreates the original Colab exploration in a cleaner local workflow. The goal for this first milestone is binary classification: each CCLE cell line is labeled as TP53-mutant or TP53-wild-type, then gene expression is used to rank features and train a first simple model.

## 1. Load Shared Project Code

The notebook imports the same modules used by the command-line scripts. That way, the exploratory notebook and the reproducible pipeline do not drift apart as the project grows.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tp53_baseline.config import BaselineConfig
from tp53_baseline.data import (
    TARGET_COLUMN,
    build_labeled_dataset,
    get_numeric_feature_matrix,
    get_tp53_related_columns,
    load_expression_data,
    load_mutation_data,
    summarize_dataset,
)
from tp53_baseline.feature_selection import compute_feature_ranking, exclude_columns_by_name, select_top_features
from tp53_baseline.model_training import train_top_features_model

## 2. Configure Local Paths

The raw CCLE CSVs are intentionally kept outside git because they are large. The defaults below match the local files used for the first baseline run, and each path can be overridden with environment variables.

In [2]:
expression_path = Path(os.environ.get("TP53_EXPRESSION_CSV", "/Users/v_angelov/Downloads/CCLE_expression_full.csv"))
mutation_path = Path(os.environ.get("TP53_MUTATION_CSV", "/Users/v_angelov/Downloads/CCLE_mutations.csv"))
output_dir = Path(os.environ.get("TP53_OUTPUT_DIR", REPO_ROOT / "outputs" / "baseline"))
model_output_dir = Path(os.environ.get("TP53_MODEL_OUTPUT_DIR", REPO_ROOT / "outputs" / "models" / "top500_logistic"))
top_n = int(os.environ.get("TP53_TOP_N", "500"))
exclude_prefix = os.environ.get("TP53_EXCLUDE_PREFIX", "TP53")

expression_path, mutation_path, output_dir, model_output_dir

(PosixPath('/Users/v_angelov/Downloads/CCLE_expression_full.csv'),
 PosixPath('/Users/v_angelov/Downloads/CCLE_mutations.csv'),
 PosixPath('/Users/v_angelov/GroupProj/TP53-MUTATIONS/outputs/baseline'),
 PosixPath('/Users/v_angelov/GroupProj/TP53-MUTATIONS/outputs/models/top500_logistic'))

## 3. Inspect The Raw Tables

The expression matrix has one row per cell line and many gene-expression columns. The mutation table has one row per mutation event, so we reduce it to a per-cell-line TP53 label in the next step.

In [3]:
expression_df = load_expression_data(expression_path)
mutation_df = load_mutation_data(mutation_path)

expression_df.shape, mutation_df.shape

((1406, 53971), (1235466, 2))

In [4]:
expression_df.head()

,DepMap_ID,TSPAN6 (ENSG00000000003),TNMD (ENSG00000000005),DPM1 (ENSG00000000419),SCYL3 (ENSG00000000457),C1orf112 (ENSG00000000460),FGR (ENSG00000000938),CFH (ENSG00000000971),FUCA2 (ENSG00000001036),GCLC (ENSG00000001084),...,ENSG00000288714,ENSG00000288717,ENSG00000288718,ENSG00000288719,ENSG00000288720,ENSG00000288721,ENSG00000288722,ENSG00000288723,ENSG00000288724,ENSG00000288725
0,ACH-001113,4.331992,0.000000,7.364397,2.792855,4.470537,0.028569,1.226509,3.042644,6.499686,...,0.000000,0.536053,0.000000,0.028569,0.176323,0.992768,2.794936,0.000000,0.0,0.000000
1,ACH-001289,4.566815,0.584963,7.106537,2.543496,3.504620,0.000000,0.189034,3.813525,4.221104,...,0.000000,0.879706,0.000000,0.014355,0.014355,0.432959,2.972693,0.056584,0.0,0.070389
2,ACH-001339,3.150560,0.000000,7.379032,2.333424,4.227279,0.056584,1.310340,6.687061,3.682573,...,0.028569,0.000000,0.084064,0.000000,0.097611,0.367371,1.695994,0.084064,0.0,0.000000
3,ACH-001538,5.085340,0.000000,7.154109,2.545968,3.084064,0.000000,5.868143,6.165309,4.489928,...,0.000000,0.000000,0.070389,0.000000,0.176323,0.411426,3.921246,0.028569,0.0,0.000000
4,ACH-000242,6.729145,0.000000,6.537607,2.456806,3.867896,0.799087,7.208381,5.569856,7.127014,...,0.000000,0.000000,0.201634,0.028569,0.137504,0.678072,4.418190,0.000000,0.0,0.000000


In [5]:
mutation_df.head()

,Hugo_Symbol,DepMap_ID
0,VPS13D,ACH-000001
1,AADACL4,ACH-000001
2,IFNLR1,ACH-000001
3,TMEM57,ACH-000001
4,ZSCAN20,ACH-000001


In [6]:
{
    "expression_columns_preview": expression_df.columns[:10].tolist(),
    "mutation_columns": mutation_df.columns.tolist(),
    "overlapping_depmap_ids": len(set(expression_df["DepMap_ID"]).intersection(set(mutation_df["DepMap_ID"]))),
}

{'expression_columns_preview': ['DepMap_ID',
  'TSPAN6 (ENSG00000000003)',
  'TNMD (ENSG00000000005)',
  'DPM1 (ENSG00000000419)',
  'SCYL3 (ENSG00000000457)',
  'C1orf112 (ENSG00000000460)',
  'FGR (ENSG00000000938)',
  'CFH (ENSG00000000971)',
  'FUCA2 (ENSG00000001036)',
  'GCLC (ENSG00000001084)'],
 'mutation_columns': ['Hugo_Symbol', 'DepMap_ID'],
 'overlapping_depmap_ids': 1402}

## 4. Build Binary TP53 Labels

A cell line is labeled `1` if it has at least one mutation row where `Hugo_Symbol == "TP53"`. All other expression samples are labeled `0` for this first binary task.

In [7]:
tp53_labels = mutation_df[mutation_df["Hugo_Symbol"] == "TP53"][["DepMap_ID"]].drop_duplicates()
tp53_labels["TP53_status"] = 1

tp53_labels.head()

,DepMap_ID,TP53_status
95,ACH-000001,1
581,ACH-000003,1
800,ACH-000004,1
1080,ACH-000005,1
1328,ACH-000006,1


In [8]:
dataset = build_labeled_dataset(expression_df, mutation_df)

dataset[["DepMap_ID", TARGET_COLUMN]].head()

,DepMap_ID,TP53_status
0,ACH-001113,1
1,ACH-001289,0
2,ACH-001339,1
3,ACH-001538,1
4,ACH-000242,0


In [9]:
dataset[TARGET_COLUMN].value_counts().rename(index={0: "WT", 1: "TP53-mutant"})

TP53_status
TP53-mutant    882
WT             524
Name: count, dtype: int64

## 5. Prepare Numeric Expression Features

We keep numeric expression columns only. Before ranking, we remove columns whose names contain `TP53`, including the direct `TP53` expression column and related TP53-family columns. This avoids giving the model an overly direct signal from the gene whose mutation status we are trying to predict.

In [10]:
features = get_numeric_feature_matrix(dataset)
target = dataset[TARGET_COLUMN]
tp53_related_columns = get_tp53_related_columns(list(features.columns), exclude_prefix)
filtered_features = exclude_columns_by_name(features, tp53_related_columns)

summary = summarize_dataset(
    expression_df=expression_df,
    mutation_df=mutation_df,
    dataset=dataset,
    numeric_features=features,
    filtered_features=filtered_features,
    excluded_columns=tp53_related_columns,
)
summary

{'expression_samples': 1406,
 'mutation_samples': 1771,
 'overlap_samples': 1402,
 'merged_samples': 1406,
 'tp53_positive_samples': 882,
 'tp53_negative_samples': 524,
 'positive_rate': 0.627312,
 'numeric_feature_count': 53970,
 'excluded_tp53_related_feature_count': 20,
 'remaining_feature_count': 53950}

In [11]:
tp53_related_columns[:20]

['TP53BP1 (ENSG00000067369)',
 'TP53INP2 (ENSG00000078804)',
 'TP53I3 (ENSG00000115129)',
 'TP53AIP1 (ENSG00000120471)',
 'TP53TG5 (ENSG00000124251)',
 'TP53 (ENSG00000141510)',
 'TP53BP2 (ENSG00000143514)',
 'TP53INP1 (ENSG00000164938)',
 'TP53I13 (ENSG00000167543)',
 'TP53RK (ENSG00000172315)',
 'TP53I11 (ENSG00000175274)',
 'TP53TG1 (ENSG00000182165)',
 'TP53TG3 (ENSG00000183632)',
 'TP53TG3D (ENSG00000205456)',
 'TP53TG3C (ENSG00000205457)',
 'TP53TG3GP (ENSG00000261274)',
 'TP53TG3B (ENSG00000261509)',
 'TP53TG3HP (ENSG00000269622)',
 'TP53TG3E (ENSG00000275034)',
 'TP53TG3F (ENSG00000278848)']

## 6. Rank Genes And Save The Top 500

The feature ranking follows the original Colab idea: compare mutant versus wild-type expression using absolute mean difference, absolute Cohen's d effect size, and Welch t-test p-value. For this first artifact, features are sorted by absolute effect size.

In [12]:
ranking_df = compute_feature_ranking(filtered_features, target)
top_features_df = select_top_features(ranking_df, top_n)

output_dir.mkdir(parents=True, exist_ok=True)
summary_path = output_dir / "dataset_summary.json"
ranking_path = output_dir / "feature_ranking.csv"
top_features_path = output_dir / f"top_{top_n}_features.csv"
excluded_columns_path = output_dir / "excluded_tp53_related_features.txt"

summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
ranking_df.to_csv(ranking_path, index=False)
top_features_df.to_csv(top_features_path, index=False)
excluded_columns_path.write_text("\n".join(tp53_related_columns) + "\n", encoding="utf-8")

ranking_df.head(20)

,feature,mean_diff,effect_size,p_value
0,EDA2R (ENSG00000131080),1.617989,1.386310,4.384187e-85
1,MDM2 (ENSG00000135679),1.350453,1.125455,8.814572e-66
2,CDKN1A (ENSG00000124762),1.847041,1.092069,1.842553e-76
3,ZMAT3 (ENSG00000172667),1.043897,1.079944,6.363047e-65
4,RPS27L (ENSG00000185088),1.079684,1.033720,2.084119e-66
5,AC025423.2 (ENSG00000256664),1.047966,0.909592,1.305049e-48
6,LNCTAM34A (ENSG00000234546),0.696622,0.902265,7.225902e-51
7,MIR34AHG (ENSG00000228526),0.899241,0.854847,1.247433e-45
8,PTCHD4 (ENSG00000244694),0.632784,0.838350,9.671181e-38
9,SPATA18 (ENSG00000163071),0.790334,0.835055,1.297944e-36


## 7. First Model: Stratified Baseline vs Logistic Regression

Now we train on the saved top-500 feature set. The stratified dummy classifier is the sanity check: it predicts according to the training class distribution. Logistic regression is the first real model, using standardized expression values and class balancing.

Important caveat: the top-500 list was selected using the full dataset before this train/test split. These metrics are useful for checking whether there is signal, but they are not yet a final unbiased estimate. Later, feature selection should happen inside each training fold.

In [13]:
training_results = train_top_features_model(
    expression_csv=expression_path,
    mutation_csv=mutation_path,
    top_features_csv=top_features_path,
    output_dir=model_output_dir,
    test_size=0.25,
    random_state=42,
)

training_results["metrics"]

{'dataset': {'samples': 1406,
  'features': 500,
  'positive_samples': 882,
  'negative_samples': 524,
  'test_size': 0.25,
  'random_state': 42,
  'model': "StandardScaler + LogisticRegression(C=0.1, class_weight='balanced', solver='liblinear')",
  'note': 'Top features were selected before this split, so treat scores as exploratory.'},
 'dummy_stratified': {'accuracy': 0.517045,
  'balanced_accuracy': 0.483265,
  'precision': 0.615385,
  'recall': 0.615385,
  'f1': 0.615385,
  'roc_auc': 0.483265,
  'confusion_matrix': {'labels': [0, 1], 'values': [[46, 85], [85, 136]]}},
 'logistic_regression_top500': {'accuracy': 0.849432,
  'balanced_accuracy': 0.839677,
  'precision': 0.881818,
  'recall': 0.877828,
  'f1': 0.879819,
  'roc_auc': 0.898484,
  'confusion_matrix': {'labels': [0, 1], 'values': [[105, 26], [27, 194]]}}}

In [14]:
metrics = training_results["metrics"]
comparison_df = pd.DataFrame(
    [
        {"model": "dummy_stratified", **metrics["dummy_stratified"]},
        {"model": "logistic_regression_top500", **metrics["logistic_regression_top500"]},
    ]
).drop(columns=["confusion_matrix"])

comparison_df

,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
0,dummy_stratified,0.517045,0.483265,0.615385,0.615385,0.615385,0.483265
1,logistic_regression_top500,0.849432,0.839677,0.881818,0.877828,0.879819,0.898484


## 8. Inspect Model Coefficients

These coefficients are not a biological conclusion by themselves, but they give us a first look at which selected genes are most influential in the linear classifier.

In [15]:
coefficients_df = pd.read_csv(training_results["coefficients_path"])
coefficients_df.head(20)

,feature,coefficient,abs_coefficient
0,FRRS1 (ENSG00000156869),0.497066,0.497066
1,FDXR (ENSG00000161513),-0.456015,0.456015
2,LAD1 (ENSG00000159166),-0.419052,0.419052
3,LINC01980 (ENSG00000225548),0.395218,0.395218
4,GPATCH1 (ENSG00000076650),-0.393583,0.393583
5,RPS27L (ENSG00000185088),-0.382350,0.382350
6,AC083801.2 (ENSG00000260377),0.380651,0.380651
7,AL079303.1 (ENSG00000258661),0.353175,0.353175
8,SYTL1 (ENSG00000142765),0.350232,0.350232
9,MRPL49 (ENSG00000149792),0.342512,0.342512
